# Lab 04 — CNNs, residual connections and depth

**Dataset:** [ylecun/mnist](https://huggingface.co/datasets/ylecun/mnist). We compare a small fully-connected model with a CNN and then test whether a residual block changes optimization behavior. Validation data guide comparisons; the test split is reserved for one final evaluation.

## The math behind the experiment

A convolution applies a small kernel across local image regions. In simplified notation,

$$
y_{i,j}=\sum_{u,v}K_{u,v}x_{i+u,j+v}+b.
$$

The same kernel is reused at many image locations. That is **parameter sharing**: the model can learn an edge detector once and use it across the image.

A residual block instead learns a correction to an existing representation:

$$
y=x+F(x).
$$

The shortcut gives gradients and information a direct path through the network, which helps very deep models train.

In [ ]:
!pip -q install datasets torch torchvision pandas matplotlib
from datasets import load_dataset
import numpy as np,pandas as pd,torch
from torch import nn
from torch.utils.data import TensorDataset,DataLoader
tr=load_dataset("ylecun/mnist",split="train[:4000]"); te=load_dataset("ylecun/mnist",split="test[:1000]")
def make(ds): return torch.tensor(np.array([np.array(x) for x in ds["image"]])/255.,dtype=torch.float32).unsqueeze(1),torch.tensor(ds["label"])
Xall,yall=make(tr); Xte,yte=make(te)
split_gen=torch.Generator().manual_seed(17)
perm=torch.randperm(len(Xall),generator=split_gen)
n_val=max(1,int(.2*len(Xall)))
val_idx,train_idx=perm[:n_val],perm[n_val:]
Xtr,ytr=Xall[train_idx],yall[train_idx]
Xval,yval=Xall[val_idx],yall[val_idx]
print("train",Xtr.shape,"validation",Xval.shape,"test",Xte.shape)

## Step 1 — Prediction

CNNs should need fewer parameters than a dense network for image structure because the same local filter is reused across positions. Expect competitive or better accuracy despite fewer parameters.

In [ ]:
class Dense(nn.Module):
 def __init__(self): super().__init__(); self.net=nn.Sequential(nn.Flatten(),nn.Linear(784,64),nn.ReLU(),nn.Linear(64,10))
 def forward(self,x): return self.net(x)
class CNN(nn.Module):
 def __init__(self): super().__init__(); self.net=nn.Sequential(nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(32,10))
 def forward(self,x): return self.net(x)
def fit(m,epochs=3):
 opt=torch.optim.Adam(m.parameters(),lr=1e-3); loss=nn.CrossEntropyLoss(); dl=DataLoader(TensorDataset(Xtr,ytr),64,shuffle=True)
 for _ in range(epochs):
  for xb,yb in dl: opt.zero_grad(); loss(m(xb),yb).backward(); opt.step()
 with torch.no_grad(): acc=(m(Xval).argmax(1)==yval).float().mean().item()
 return sum(p.numel() for p in m.parameters()),acc
rows=[]
for n,m in [("dense",Dense()),("cnn",CNN())]: rows.append([n,*fit(m)])
pd.DataFrame(rows,columns=["model","parameters","validation_accuracy"])

## Step 2 — Receptive-field intervention

Replace the CNN's local 3×3 kernels with a 1×1 convolution while keeping the training protocol fixed. Predict that the modified model should lose the spatial inductive bias.

In [ ]:
class CNN1x1(nn.Module):
 def __init__(self): super().__init__(); self.net=nn.Sequential(nn.Conv2d(1,16,1),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(16,32,1),nn.ReLU(),nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(32,10))
 def forward(self,x): return self.net(x)
rows.append(["cnn_1x1",*fit(CNN1x1())]); pd.DataFrame(rows,columns=["model","parameters","validation_accuracy"])

## Read the graph — accuracy must be considered with model size

The plot compares parameter count and validation accuracy for the dense baseline, spatial CNN, and 1×1-convolution intervention. The 1×1 model removes most local spatial mixing, while the 3×3 model can learn patterns across neighboring pixels.

Do not conclude that one architecture is universally best from one short run. Check the parameter budget, training time, random seed, and variability before making a stronger claim.

In [ ]:
import matplotlib.pyplot as plt

model_results = pd.DataFrame(rows, columns=["model", "parameters", "validation_accuracy"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(model_results["model"], model_results["parameters"])
axes[0].set_title("Model size")
axes[0].set_ylabel("Trainable parameters")
axes[0].tick_params(axis="x", rotation=15)
axes[0].grid(axis="y", alpha=.25)

axes[1].bar(model_results["model"], model_results["validation_accuracy"])
axes[1].set_title("Validation accuracy")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=15)
axes[1].grid(axis="y", alpha=.25)
fig.tight_layout()
plt.show()

## Step 3 — Compare plain, residual, and dense-style blocks

The residual connection adds the input back to the transformed features. A DenseNet-style block instead **concatenates** new features with earlier features, so later layers can reuse them directly.

We compare a plain block, an identity-skip residual block, and a small dense-style block. The dense block uses a growth rate of 5 so that total trainable parameter counts stay within about 1% of the other two models. All variants use the same training/validation split, optimizer, epoch budget, and minibatch order per seed. Plain and residual variants share exactly the same initialization; the dense model has different layer shapes, so it is initialized from the same seed rather than identical tensors.

This is a short teaching comparison, not a full ResNet/DenseNet benchmark.

In [ ]:
import torch.nn.functional as F

class ConvBlockModel(nn.Module):
    def __init__(self,residual=False):
        super().__init__()
        self.residual=residual
        self.stem=nn.Conv2d(1,16,3,padding=1)
        self.conv1=nn.Conv2d(16,16,3,padding=1)
        self.conv2=nn.Conv2d(16,16,3,padding=1)
        self.head=nn.Sequential(
            nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(32,10)
        )
    def forward(self,x):
        x=F.relu(self.stem(x))
        skip=x
        x=F.relu(self.conv1(x))
        x=self.conv2(x)
        if self.residual:
            x=x+skip
        x=F.relu(x)
        return self.head(x)

class DenseStyleModel(nn.Module):
    def __init__(self,growth=5):
        super().__init__()
        self.stem=nn.Conv2d(1,16,3,padding=1)
        self.dense1=nn.Conv2d(16,growth,3,padding=1)
        self.dense2=nn.Conv2d(16+growth,growth,3,padding=1)
        dense_channels=16+2*growth
        self.head=nn.Sequential(
            nn.MaxPool2d(2),
            nn.Conv2d(dense_channels,32,3,padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(32,10)
        )
    def forward(self,x):
        x=F.relu(self.stem(x))
        new1=F.relu(self.dense1(x))
        new2=F.relu(self.dense2(torch.cat([x,new1],dim=1)))
        features=torch.cat([x,new1,new2],dim=1)
        return self.head(features)

def run_cnn_variant(kind,seed,epochs=3):
    torch.manual_seed(seed)
    if kind=="plain":
        model=ConvBlockModel(residual=False)
    elif kind=="residual":
        model=ConvBlockModel(residual=True)
    elif kind=="dense-style":
        model=DenseStyleModel(growth=5)
    else:
        raise ValueError(kind)
    n_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
    optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)
    criterion=nn.CrossEntropyLoss()
    generator=torch.Generator().manual_seed(seed+100)
    loader=DataLoader(TensorDataset(Xtr,ytr),batch_size=64,shuffle=True,generator=generator)
    history=[]
    for epoch in range(1,epochs+1):
        model.train(); losses=[]
        for xb,yb in loader:
            optimizer.zero_grad()
            loss=criterion(model(xb),yb)
            loss.backward(); optimizer.step()
            losses.append(loss.item())
        model.eval()
        with torch.no_grad():
            logits=model(Xval)
            validation_loss=criterion(logits,yval).item()
            validation_acc=(logits.argmax(1)==yval).float().mean().item()
        history.append({
            "seed":seed,"model":kind,"epoch":epoch,
            "train_loss":float(np.mean(losses)),
            "validation_loss":validation_loss,"validation_accuracy":validation_acc,
            "parameters":n_params
        })
    return history

cnn_rows=[]
for seed in [0,1,2]:
    # Same seed and minibatch order for all variants; plain/residual also share exact shapes.
    for kind in ["plain","residual","dense-style"]:
        cnn_rows.extend(run_cnn_variant(kind,seed))
cnn_results=pd.DataFrame(cnn_rows)
param_counts=cnn_results.groupby("model")["parameters"].first()
relative_spread=(param_counts.max()-param_counts.min())/param_counts.mean()
assert relative_spread<0.02, f"Parameter budgets differ by more than 2%: {param_counts.to_dict()}"
print("Trainable parameters:",param_counts.to_dict())
print(f"Relative parameter-count spread: {relative_spread:.2%}")
display(cnn_results[cnn_results["epoch"]==3][["seed","model","parameters","validation_loss","validation_accuracy"]])

In [ ]:
import matplotlib.pyplot as plt

fig,axes=plt.subplots(1,2,figsize=(13,4.8))
for model_name in ["plain","residual","dense-style"]:
    part=cnn_results[cnn_results["model"]==model_name]
    summary=part.groupby("epoch")["train_loss"].agg(["mean","std"]).reset_index()
    axes[0].errorbar(summary["epoch"],summary["mean"],yerr=summary["std"].fillna(0),
                     marker="o",capsize=4,label=model_name)
axes[0].set_title("Training loss by epoch (3 seeds)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Training cross-entropy")
axes[0].legend()
axes[0].grid(alpha=.25)

final=cnn_results[cnn_results["epoch"]==cnn_results["epoch"].max()]
summary=final.groupby("model")["validation_accuracy"].agg(["mean","std"]).reindex(["plain","residual","dense-style"])
axes[1].bar(summary.index,summary["mean"],yerr=summary["std"].fillna(0),capsize=4)
axes[1].set_title("Final validation accuracy (3 seeds)")
axes[1].set_ylabel("Accuracy (mean ± sample SD)")
axes[1].set_ylim(0,1)
axes[1].tick_params(axis="x",rotation=15)
axes[1].grid(axis="y",alpha=.25)
fig.tight_layout()
plt.show()

## Final check on the untouched test split

Use the three-seed validation comparison to choose one architecture. Then train that selected architecture on the combined training-plus-validation data for the same three-epoch budget and evaluate it once on the untouched test split. Do not use the final test score to change the architecture or training settings.


In [ ]:
final_epoch=int(cnn_results["epoch"].max())
chosen_kind=cnn_results[cnn_results["epoch"]==final_epoch].groupby("model")["validation_accuracy"].mean().idxmax()
torch.manual_seed(42)
if chosen_kind=="plain":
    final_model=ConvBlockModel(residual=False)
elif chosen_kind=="residual":
    final_model=ConvBlockModel(residual=True)
else:
    final_model=DenseStyleModel(growth=5)
final_optimizer=torch.optim.Adam(final_model.parameters(),lr=1e-3)
criterion=nn.CrossEntropyLoss()
final_gen=torch.Generator().manual_seed(142)
final_loader=DataLoader(TensorDataset(Xall,yall),batch_size=64,shuffle=True,generator=final_gen)
for _ in range(3):
    final_model.train()
    for xb,yb in final_loader:
        final_optimizer.zero_grad()
        criterion(final_model(xb),yb).backward()
        final_optimizer.step()
final_model.eval()
with torch.no_grad():
    final_logits=final_model(Xte)
    final_test_accuracy=float((final_logits.argmax(1)==yte).float().mean().item())
print({"validation_selected_architecture":chosen_kind,"final_test_accuracy":final_test_accuracy})


### Interpretation and answer key

The parameter-count assertion should pass because the dense-style block uses a small growth rate; its concatenation changes the feature flow without greatly increasing the total parameter budget. Residual learning adds the input to the transformed features, while dense-style connectivity concatenates earlier features for reuse. They are not the same operation.

Compare loss curves and validation accuracy only after checking seed variability. If one model does not win in this short run, that does not disprove its architectural idea: the network is shallow, the dataset subset is small, and training budget matters. The dense-style model has different layer shapes, so the same seed does not give it identical tensors to the other variants.

**Research extension:** repeat with at least five seeds and a deeper matched pair. Report wall-clock time and activation memory as well as accuracy; dense connectivity can increase memory traffic even when parameter counts are close.

## Conclusion / answer key

Expected: the CNN should exploit local spatial structure more effectively than the dense baseline at comparable scale. The 1×1 intervention removes most local spatial mixing, so any drop is evidence for the convolutional inductive bias rather than merely “more layers.”

### Research extension
Compare residual vs plain blocks at equal depth and parameter count, measuring optimization curves and final accuracy.